# odseki_vector_map.mbtiles — feature metadata

Questions:
1. What zoom levels are stored in the file and how many tiles per level?
2. What property keys exist on features in the `odsek` layer?
3. Do all features have the same set of keys?
4. What are the value types and distributions per key?
5. Are there any null/missing/empty values?

---

**What is a zoom level?**  
The .mbtiles format slices the map into a grid of square tiles at each zoom level:
- Zoom 0: 1 tile covers the entire world
- Zoom 1: 4 tiles (2×2 grid)
- Zoom N: 4ᴺ tiles (2ᴺ × 2ᴺ grid) — each tile covers a smaller geographic area

Only tiles that actually contain data are stored (sparse). Higher zoom = more tiles = finer geographic
resolution = more detail per tile. For decoding feature metadata, any zoom that covers the whole country
works; we use the lowest such zoom to minimise tile count and avoid duplicates from boundary overlap.

In [1]:
import sqlite3, gzip, struct
from collections import Counter
from pathlib import Path
import pandas as pd

BASE = Path("..").resolve()
MBTILES_PATH = BASE / "data" / "odseki_vector_map.mbtiles"

print(f"mbtiles: {MBTILES_PATH}")

mbtiles: /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/odseki_vector_map.mbtiles


## 1. Zoom levels and tile counts

In [2]:
conn = sqlite3.connect(str(MBTILES_PATH))
cur  = conn.cursor()

# Tile counts per zoom level
cur.execute("SELECT zoom_level, count(*) FROM tiles GROUP BY zoom_level ORDER BY zoom_level")
zoom_rows = cur.fetchall()

# mbtiles metadata table (min/max zoom, bounds, format, ...)
cur.execute("SELECT name, value FROM metadata")
meta_rows = {row[0]: row[1] for row in cur.fetchall()}

conn.close()

print("=== mbtiles metadata ===")
for k, v in sorted(meta_rows.items()):
    print(f"  {k:<20} {v}")

print()
print("=== Tile counts per zoom level ===")
print(f"{'Zoom':>6}  {'Tiles stored':>14}  {'Max possible tiles':>20}  {'Coverage':>10}")
print("-" * 56)
for zoom, count in zoom_rows:
    max_tiles = 4 ** zoom
    coverage  = count / max_tiles * 100
    print(f"  {zoom:>4}  {count:>14,}  {max_tiles:>20,}  {coverage:>9.4f}%")

print()
print(f"Total zoom levels in file: {len(zoom_rows)}  (zoom {zoom_rows[0][0]} – {zoom_rows[-1][0]})")
print(f"Total tiles stored:        {sum(c for _, c in zoom_rows):,}")

=== mbtiles metadata ===
  antimeridian_adjusted_bounds 13.375463,45.421923,16.594469,46.876646
  bounds               13.375463,45.421923,16.594469,46.876646
  center               13.765869,46.050360,14
  description          /home/valentin/uporabnik/hackathon2026/analiza_sestoji_odseki/map_vector_finalll.mbtiles
  format               pbf
  generator            tippecanoe v2.80.0
  generator_options    tippecanoe -o /home/valentin/uporabnik/hackathon2026/analiza_sestoji_odseki/map_vector_finalll.mbtiles '--layer=odsek' --drop-densest-as-needed /home/valentin/uporabnik/hackathon2026/analiza_sestoji_odseki/odseki/odseki_map_final.geojson --force
  json                 {"vector_layers":[{"id":"odsek","description":"","minzoom":0,"maxzoom":14,"fields":{"ggo_naziv":"String","odsek":"String"}}],"tilestats":{"layerCount":1,"layers":[{"layer":"odsek","count":53254,"geometry":"Polygon","attributeCount":2,"attributes":[{"attribute":"ggo_naziv","count":14,"type":"string","values":["BLED       

In [3]:
# Pick the lowest zoom that has more than 1 tile (zoom 0–6 is just 1 tile = whole world)
# We want the lowest zoom where all segments are represented without too many tile-boundary duplicates
# Zoom 11 (154 tiles) is the same level the server uses for the bbox cache
DECODE_ZOOM = 11
print(f"Using zoom {DECODE_ZOOM} for feature extraction (same as server bbox cache)")

Using zoom 11 for feature extraction (same as server bbox cache)


## 2. MVT decoder and feature extraction

In [4]:
# ── Minimal MVT decoder (no external deps) ──────────────────────────────────

def _read_varint(data, pos):
    result = shift = 0
    while pos < len(data):
        b = data[pos]; pos += 1
        result |= (b & 0x7F) << shift
        if not (b & 0x80): return result, pos
        shift += 7
    return result, pos

def _zigzag(n): return (n >> 1) ^ -(n & 1)

def _unpack_varints(data):
    values, pos = [], 0
    while pos < len(data):
        v, pos = _read_varint(data, pos)
        values.append(v)
    return values

def _decode_value(data):
    pos = 0
    while pos < len(data):
        tw, pos = _read_varint(data, pos)
        fn, wt = tw >> 3, tw & 0x7
        if wt == 0:
            v, pos = _read_varint(data, pos)
            if fn in (4, 5): return v
            if fn == 6: return _zigzag(v)
            if fn == 7: return bool(v)
        elif wt == 2:
            l, pos = _read_varint(data, pos)
            chunk = data[pos:pos+l]; pos += l
            if fn == 1: return chunk.decode('utf-8', errors='replace')
        elif wt == 1:
            v = struct.unpack_from('<d', data, pos)[0]; pos += 8
            return v if fn == 3 else None
        elif wt == 5:
            v = struct.unpack_from('<f', data, pos)[0]; pos += 4
            return v if fn == 2 else None
    return None

def _yield_layer_features(data, target_layer):
    """Yield property dicts for all features in target_layer."""
    keys, raw_vals, raw_feats, name = [], [], [], ''
    pos = 0
    while pos < len(data):
        tw, pos = _read_varint(data, pos)
        fn, wt = tw >> 3, tw & 0x7
        if wt == 2:
            l, pos = _read_varint(data, pos)
            chunk = data[pos:pos+l]; pos += l
            if fn == 1: name = chunk.decode('utf-8', errors='replace')
            elif fn == 2: raw_feats.append(chunk)
            elif fn == 3: keys.append(chunk.decode('utf-8', errors='replace'))
            elif fn == 4: raw_vals.append(_decode_value(chunk))
        elif wt == 0: _, pos = _read_varint(data, pos)
        elif wt == 1: pos += 8
        elif wt == 5: pos += 4
    if name != target_layer:
        return
    for fd in raw_feats:
        tags_raw = None; fp = 0
        while fp < len(fd):
            tw, fp = _read_varint(fd, fp)
            fn2, wt2 = tw >> 3, tw & 0x7
            if wt2 == 0: _, fp = _read_varint(fd, fp)
            elif wt2 == 2:
                l, fp = _read_varint(fd, fp)
                chunk = fd[fp:fp+l]; fp += l
                if fn2 == 2: tags_raw = chunk
            elif wt2 == 1: fp += 8
            elif wt2 == 5: fp += 4
        if not tags_raw: continue
        tag_ints = _unpack_varints(tags_raw)
        props = {}
        for k in range(0, len(tag_ints)-1, 2):
            ki, vi = tag_ints[k], tag_ints[k+1]
            if ki < len(keys) and vi < len(raw_vals):
                props[keys[ki]] = raw_vals[vi]
        yield props


def extract_all_features(mbtiles_path, zoom, layer='odsek'):
    """Return list of property dicts for all features at the given zoom."""
    conn = sqlite3.connect(str(mbtiles_path))
    cur  = conn.cursor()
    cur.execute("SELECT tile_data FROM tiles WHERE zoom_level=?", (zoom,))
    tile_rows = cur.fetchall()
    conn.close()

    all_props = []
    for (tile_data,) in tile_rows:
        if not tile_data:
            continue
        raw = bytes(tile_data)
        if raw[:2] == b'\x1f\x8b':
            raw = gzip.decompress(raw)
        pos = 0
        while pos < len(raw):
            tw, pos = _read_varint(raw, pos)
            fn, wt = tw >> 3, tw & 0x7
            if wt == 2:
                l, pos = _read_varint(raw, pos)
                chunk = raw[pos:pos+l]; pos += l
                if fn == 3:
                    all_props.extend(_yield_layer_features(chunk, layer))
            elif wt == 0: _, pos = _read_varint(raw, pos)
            elif wt == 1: pos += 8
            elif wt == 5: pos += 4
    return all_props


print(f"Extracting features from zoom {DECODE_ZOOM}...")
features = extract_all_features(MBTILES_PATH, DECODE_ZOOM)
print(f"Total raw features decoded: {len(features):,}")
print(f"(includes duplicates from tile-boundary overlap — same polygon in adjacent tiles)")

Extracting features from zoom 11...
Total raw features decoded: 63,460
(includes duplicates from tile-boundary overlap — same polygon in adjacent tiles)


## 3. What keys exist and how consistently are they present?

In [5]:
n = len(features)

key_presence = Counter()
for props in features:
    key_presence.update(props.keys())

print(f"{'Key':<30} {'Present':>10}  {'% of features':>14}")
print("-" * 58)
for key, count in sorted(key_presence.items(), key=lambda x: -x[1]):
    flag = "" if count == n else "  <- MISSING in some"
    print(f"{key:<30} {count:>10,}  {count/n*100:>13.1f}%{flag}")

print(f"\nTotal unique keys: {len(key_presence)}")
print(f"Total features:    {n:,}")

Key                               Present   % of features
----------------------------------------------------------
odsek                              63,460          100.0%
ggo_naziv                          63,460          100.0%

Total unique keys: 2
Total features:    63,460


In [6]:
# Is the key set uniform across all features?
key_sets = Counter(frozenset(p.keys()) for p in features)

print(f"Distinct key-set variants: {len(key_sets)}")
for ks, cnt in key_sets.most_common():
    print(f"  {cnt:>8,} features — keys: {sorted(ks)}")

Distinct key-set variants: 1
    63,460 features — keys: ['ggo_naziv', 'odsek']


## 4. Per-key statistics — types, unique values, nulls/blanks

In [7]:
df = pd.DataFrame(features)
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)

Shape: (63460, 2)

Column dtypes:
odsek        str
ggo_naziv    str
dtype: object


In [8]:
print(f"{'Key':<30} {'dtype':<12} {'n_unique':>9} {'null':>8} {'blank_str':>10}")
print("-" * 72)

for col in sorted(df.columns):
    s = df[col]
    n_null  = s.isna().sum()
    n_blank = (s.astype(str).str.strip() == '').sum() if s.dtype == object else 0
    print(f"{col:<30} {str(s.dtype):<12} {s.nunique():>9,} {n_null:>8,} {n_blank:>10,}")

Key                            dtype         n_unique     null  blank_str
------------------------------------------------------------------------
ggo_naziv                      str                 14        0          0
odsek                          str             35,687        0          0


## 5. Value distribution per key

In [9]:
MAX_SHOW = 20

for col in sorted(df.columns):
    s = df[col]
    n_unique = s.nunique()
    print(f"\n{'='*60}")
    print(f"  {col}  (dtype={s.dtype}, unique={n_unique:,})")
    print(f"{'='*60}")

    if pd.api.types.is_numeric_dtype(s):
        print(s.describe().round(4).to_string())
        if n_unique <= MAX_SHOW:
            print("\nAll values:")
            print(s.value_counts().sort_index().to_string())
    else:
        vc = s.value_counts(dropna=False)
        if n_unique <= MAX_SHOW:
            print(vc.to_string())
        else:
            print(f"Top {MAX_SHOW} values:")
            print(vc.head(MAX_SHOW).to_string())
            print(f"  ... and {n_unique - MAX_SHOW} more")


  ggo_naziv  (dtype=str, unique=14)
ggo_naziv
MARIBOR                           9021
CELJE                             7207
LJUBLJANA                         6319
TOLMIN                            4960
KOČEVJE                           4890
POSTOJNA                          4199
SLOVENJ GRADEC                    4070
NOVO MESTO                        3986
KRANJ                             3550
BREŽICE                           3317
SEŽANA                            3286
MURSKA SOBOTA                     3139
NAZARJE                           2941
BLED                              2575

  odsek  (dtype=str, unique=35,687)
Top 20 values:
odsek
09050A    13
13004     13
09050B    12
03001B    12
01017B    11
03115     11
03114     11
02069     11
03001A    11
05074     11
06061     11
06045     11
07020B    11
07142     11
13071     11
13013     11
03022     10
02079     10
02016     10
02049B    10
  ... and 35667 more


## 6. Features with incomplete metadata

In [10]:
expected_keys = {k for k, cnt in key_presence.items() if cnt == n}
optional_keys = set(key_presence.keys()) - expected_keys

print(f"Keys present in ALL features:  {sorted(expected_keys)}")
print(f"Keys missing from some:        {sorted(optional_keys)}")

if optional_keys:
    for k in sorted(optional_keys):
        missing = df[k].isna().sum()
        print(f"  {k}: missing in {missing:,} / {n:,} features")
    mask = df[list(optional_keys)].isna().any(axis=1)
    print(f"\nFeatures with at least one missing optional key: {mask.sum():,}")
    print(df[mask].head(10).to_string())
else:
    print("\nAll features have exactly the same key set — metadata is uniform.")

Keys present in ALL features:  ['ggo_naziv', 'odsek']
Keys missing from some:        []

All features have exactly the same key set — metadata is uniform.


## 7. Tile-boundary duplicates

At zoom 11, polygons that straddle a tile border appear in multiple tiles.
Verify that their properties are consistent across appearances.

In [11]:
if 'odsek' in df.columns and 'ggo_naziv' in df.columns:
    df['odsek_norm'] = df['odsek'].astype(str).str.replace(' ', '', regex=False)

    group      = df.groupby(['odsek_norm', 'ggo_naziv'])
    dup_counts = group.size().rename('appearances')

    print(f"Unique (odsek, ggo) pairs:                     {len(dup_counts):,}")
    print(f"Pairs appearing in more than one tile:         {(dup_counts > 1).sum():,}")
    print(f"\nAppearance count distribution:")
    print(dup_counts.value_counts().sort_index().rename('n_pairs').to_string())

    dup_pairs = dup_counts[dup_counts > 1].index
    if len(dup_pairs):
        print(f"\nChecking property consistency across {len(dup_pairs):,} duplicated segments...")
        inconsistent = []
        check_cols = [c for c in df.columns if c != 'odsek_norm']
        for (odsek_n, ggo_n), grp in group:
            if len(grp) < 2:
                continue
            for col in check_cols:
                if grp[col].nunique(dropna=False) > 1:
                    inconsistent.append({'odsek': odsek_n, 'ggo_naziv': ggo_n, 'key': col,
                                         'values': list(grp[col].unique())})
        if inconsistent:
            print(f"  Inconsistent properties found: {len(inconsistent)}")
            print(pd.DataFrame(inconsistent).head(20).to_string())
        else:
            print("  All duplicated segments have identical properties across tiles.")
else:
    print("Columns 'odsek' and/or 'ggo_naziv' not found — skipping duplicate check.")

Unique (odsek, ggo) pairs:                     53,253
Pairs appearing in more than one tile:         9,337

Appearance count distribution:
appearances
1    43916
2     8845
3      114
4      378

Checking property consistency across 9,337 duplicated segments...
  All duplicated segments have identical properties across tiles.
